# Análise FFT + PCA — Trânsito simulado sobre frames AIA (comprimento de onda selecionável, 4 eventos)

Réplica do pipeline de análise usado para 171 Å (`Eclipse/02-Notebooks/main-fft-pca-sun-signals.ipynb`), para **qualquer canal do AIA**, nos mesmos **4 eventos** usados na análise combinada de 171 Å: `2011-06-05`, `2017-04-24`, `2017-04-30` e `2022-10-01`.

**Pipeline (por evento):**
1. Simular o trânsito de HD 189733 Ab sobre os frames COM CME e SEM CME (no canal selecionado), usando a mesma engine de trânsito do ECLIPSE (`Estrela` + `Planeta` + `Eclipse`).
2. Comparar as curvas de luz resultantes.
3. Subtrair as curvas para isolar o resíduo (assinatura da CME).
4. Analisar o espectro de frequência do resíduo via FFT.

**Pipeline (entre eventos):**
5. Correlação cruzada par a par entre os resíduos dos 4 eventos.
6. Projeção PCA (2D) dos 4 resíduos.
7. Espectros FFT dos 4 resíduos sobrepostos.

Isso replica exatamente a etapa final de `main-fft-pca-sun-signals.ipynb` (bloco `ruidos = [ruido, ruido_2, ruido_3, ruido_4]`), permitindo comparar diretamente com os mesmos resultados obtidos em 171 Å.

**Pré-requisito:** rodar antes o notebook `download-sdo-lightcurves.ipynb` (nesta mesma pasta) para o mesmo comprimento de onda que você vai selecionar aqui — ele baixa os `.fits` dos 4 eventos usados neste notebook.

### Imports

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "Eclipse", "01-Core")))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from Star.Estrela import Estrela
from Planet.Eclipse import Eclipse
from Planet.Planeta import Planeta

# GUI backend, same as the 171 A notebook: Eclipse.criarEclipse(anim=True) opens a
# blocking animation window (plt.show(block=True)) per simulate_transit() call below.
# Close each window to let the notebook move on to the next event.
%matplotlib tk


### Por que baixar vários frames por janela (não só 1)

Isso não é só para a animação visual do trânsito: em `Eclipse.criarEclipse()` (`Eclipse.py`, ramo `useFits` com `anim=True`), cada ponto de tempo `i` do trânsito simulado é mapeado para um frame específico da lista `estrela_.estrelaMatriz` via `idx_imagem = int(i * n_imgs / tam)`, e a curva de luz daquele ponto é calculada a partir *daquele* frame (`maxCurvaLuz` e a matriz transformada são recalculados por frame). Ou seja, com `anim=True` a curva de luz é montada a partir da sequência real de imagens do Sol ao longo da janela baixada — não de uma única imagem estática.

Com `anim=False` (ou com apenas 1 frame na pasta), o código usa só o primeiro frame para a curva de luz inteira.

Por isso o notebook de download baixa o intervalo completo (múltiplos frames a cada 10 min) para cada janela COM/SEM CME, e `simulate_transit()` abaixo sempre chama `eclipse_.criarEclipse(anim=True)` — exatamente como no pipeline de 171 Å.

## Selecione o comprimento de onda

Use o mesmo canal que você baixou em `download-sdo-lightcurves.ipynb`.

In [ ]:
AIA_WAVELENGTHS = [94, 131, 171, 193, 211, 304, 335, 1600, 1700, 4500]

# 171 A folders have no numeric suffix (downloaded before this convention existed);
# every other channel uses "-<wavelength>" as a suffix. Must match the download notebook.
LEGACY_NO_SUFFIX_WAVELENGTHS = {171}


def wavelength_suffix(wavelength):
    return "" if wavelength in LEGACY_NO_SUFFIX_WAVELENGTHS else f"-{wavelength}"


def select_wavelength(default=1700):
    """Prompt the user for one of the available AIA wavelengths, retrying on bad input.

    Press Enter with no input to use `default` (1700 A, the last channel used in this project).
    """
    while True:
        raw = input(
            f"Comprimento de onda AIA (Angstrom), opcoes {AIA_WAVELENGTHS} "
            f"[Enter = {default}]: "
        ).strip()
        if raw == "":
            return default
        try:
            wavelength = int(raw)
        except ValueError:
            print("Digite um numero inteiro.")
            continue
        if wavelength not in AIA_WAVELENGTHS:
            print(f"Comprimento de onda invalido. Escolha um destes: {AIA_WAVELENGTHS}")
            continue
        return wavelength


WAVELENGTH_VALUE = select_wavelength()
print(f"Comprimento de onda selecionado: {WAVELENGTH_VALUE} A")


### Nota: por que 2310 Å não está na lista de canais

O filtro UVM2 do XMM-Newton/OM (2310 Å), usado nas observações reais de HD 189733 A, não existe como canal do AIA — o instrumento tem UV apenas em 1600 e 1700 Å, pulando direto para 4500 Å (visível). Não há como "baixar 2310 Å do AIA" porque esse canal simplesmente não existe no instrumento.

A alternativa mais próxima encontrada foi o **SUIT** (Solar Ultraviolet Imaging Telescope), a bordo da missão **Aditya-L1** (ISRO), que cobre 200–400 nm em disco cheio — o filtro `BB01` (banda larga, ~200–240 nm) contém 2310 Å diretamente, ao contrário de qualquer canal do AIA. Ressalvas: (1) Aditya-L1 lançou em set/2023 e os dados científicos só foram liberados em 2025 — nenhum dos 4 eventos de referência deste projeto (2011, 2017×2, 2022) tem cobertura do SUIT; (2) acesso aos dados requer cadastro no portal PRADAN da ISRO, diferente do acesso aberto do VSO/AIA; (3) suporte via `sunpy.net.Fido` só existe a partir do SunPy 7.0. Essa opção ficou registrada aqui, mas não foi implementada — decisão de escopo em aberto.

## Parâmetros da estrela e do planeta

Mesmos valores usados no pipeline de referência de 171 Å para o sistema HD 189733 A / HD 189733 Ab — a simulação de trânsito é independente do comprimento de onda; o que muda é a matriz da estrela, construída a partir dos frames AIA de cada canal.

In [ ]:
# Star parameters (HD 189733 A)
raio_estrela_pixel = 373.0  # default (pixel)
intensidade_maxima = 240  # default
tamanho_matriz = 856  # default
raio_estrela = 0.805  # star radius relative to the solar radius
coeficiente_um = 0.377  # limb-darkening coefficient
coeficiente_dois = 0.024  # limb-darkening coefficient

# Planet parameters (HD 189733 Ab)
periodo = 2.219  # days
ecc = 0  # eccentricity
anomalia = 0  # anomaly
raio_plan_Jup = 1.138  # relative to Jupiter's radius
semi_eixo_UA = 0.031  # AU
mass_planeta = 1.138  # relative to Jupiter's radius (mass proxy used by Planeta)


## Eventos

Mesmas 4 datas e os mesmos ângulos de inclinação usados por evento no pipeline de 171 Å (`angulo_inclinacao` é alterado para os eventos de 2017-04-30 e 2022-10-01 no notebook original, mantido aqui por fidelidade).

In [ ]:
EVENTS = [
    {
        "date": "2011-06-05",
        "note": "Halo CME, GOES class B throughout, no associated flares (main reference event)",
        "angulo_inclinacao": 85.51,
    },
    {
        "date": "2017-04-24",
        "note": "CME event",
        "angulo_inclinacao": 85.51,
    },
    {
        "date": "2017-04-30",
        "note": "CME with associated flare, not halo",
        "angulo_inclinacao": 91.51,
    },
    {
        "date": "2022-10-01",
        "note": "CME with associated flare, halo",
        "angulo_inclinacao": 90.0,
    },
]


## Função reutilizável de simulação do trânsito

In [ ]:
def simulate_transit(fits_path, raio_estrela_pixel, raio_estrela, intensidade_maxima,
                      coeficiente_um, coeficiente_dois, tamanho_matriz,
                      periodo, angulo_inclinacao, ecc, anomalia,
                      raio_plan_Jup, semi_eixo_UA, mass_planeta):
    """
    Build a Estrela from AIA fits frames (fits_path, resolved under
    Sun/sdo_aia_download/) and simulate the HD 189733 Ab transit over them.
    Returns the light curve, time axis and transit duration.
    """
    estrela_ = Estrela(
        raio_estrela_pixel, raio_estrela, intensidade_maxima,
        coeficiente_um, coeficiente_dois, tamanho_matriz,
        useFits=True, fits_path=fits_path,
    )

    planeta_ = Planeta(
        semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao,
        ecc, anomalia, estrela_.getRaioSun(), mass_planeta,
    )

    eclipse_ = Eclipse(estrela_.getNx(), estrela_.getNy(), raio_estrela_pixel, estrela_, planeta_)
    eclipse_.geraTempoHoras(1)
    eclipse_.criarEclipse(anim=True)

    return {
        "estrela": estrela_,
        "planeta": planeta_,
        "eclipse": eclipse_,
        "curva_luz": np.array(eclipse_.getCurvaLuz()),
        "tempo_horas": np.array(eclipse_.getTempoHoras()),
        "tempo_transito": eclipse_.getTempoTransito(),
    }


## Simulando os 4 eventos (COM e SEM CME)

Para cada evento: constrói a estrela a partir dos frames do canal selecionado, simula o trânsito COM CME e SEM CME, e calcula o resíduo.

In [ ]:
suffix = wavelength_suffix(WAVELENGTH_VALUE)

OUTPUT_DIR = os.path.join("Products", f"{WAVELENGTH_VALUE}A")
os.makedirs(OUTPUT_DIR, exist_ok=True)

resultados = []

for event in EVENTS:
    date = event["date"]
    print(f"\n=== {date}: {event['note']} ===")

    res_cme = simulate_transit(
        f"{WAVELENGTH_VALUE}/{date}{suffix}", raio_estrela_pixel, raio_estrela, intensidade_maxima,
        coeficiente_um, coeficiente_dois, tamanho_matriz,
        periodo, event["angulo_inclinacao"], ecc, anomalia,
        raio_plan_Jup, semi_eixo_UA, mass_planeta,
    )
    res_no_cme = simulate_transit(
        f"{WAVELENGTH_VALUE}/{date}-no-cme{suffix}", raio_estrela_pixel, raio_estrela, intensidade_maxima,
        coeficiente_um, coeficiente_dois, tamanho_matriz,
        periodo, event["angulo_inclinacao"], ecc, anomalia,
        raio_plan_Jup, semi_eixo_UA, mass_planeta,
    )

    residuo = res_cme["curva_luz"] - res_no_cme["curva_luz"]

    print("Total transit time:", res_cme["tempo_transito"])

    resultados.append({
        "date": date,
        "note": event["note"],
        "tempo_horas": res_cme["tempo_horas"],
        "curva_luz_cme": res_cme["curva_luz"],
        "curva_luz_no_cme": res_no_cme["curva_luz"],
        "tempo_transito": res_cme["tempo_transito"],
        "residuo": residuo,
    })


## Curvas de luz e resíduo — por evento

Resultados salvos em `Sun/UV/Products/<comprimento_de_onda>A/` a 300 dpi, seguindo o padrão de output do repositório ECLIPSE.

In [ ]:
def plot_lightcurve_comparison(tempo_horas, curva_com_cme, curva_sem_cme, tempo_transito,
                                fig_name, output_dir=OUTPUT_DIR):
    """Plot and save the with-CME vs. without-CME light curve comparison."""
    plt.figure(figsize=(10, 5))
    plt.plot(tempo_horas, curva_com_cme, color="red", label="Light curve with CME")
    plt.plot(tempo_horas, curva_sem_cme, color="black", label="Light curve without CME")
    plt.xlabel("Time (hours)")
    plt.ylabel("Normalized intensity")
    plt.title(f"{fig_name}")
    plt.legend()
    plt.grid(True)
    plt.axis([-tempo_transito / 2, tempo_transito / 2, min(curva_com_cme) - 0.001, 1.001])
    plt.tight_layout()

    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {file_path}")
    plt.show()


def plot_residual(tempo_horas, residuo, fig_name, output_dir=OUTPUT_DIR):
    """Plot and save the residual signal (with-CME minus without-CME light curve)."""
    plt.figure(figsize=(10, 5))
    plt.plot(tempo_horas, residuo, color="darkorange", alpha=0.85,
             label=f"CME residual signal ({WAVELENGTH_VALUE} A)")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.xlabel("Time (hours)")
    plt.ylabel("Normalized flux difference")
    plt.title(f"{fig_name}")
    plt.legend()
    plt.tight_layout()

    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {file_path}")
    plt.show()


def plot_fft_spectrum(residuo, fig_name, output_dir=OUTPUT_DIR):
    """Compute and plot the FFT magnitude spectrum of a single residual signal."""
    fft_vals = np.abs(np.fft.fft(residuo))
    freqs = np.fft.fftfreq(len(residuo))
    half = len(freqs) // 2

    plt.figure(figsize=(10, 5))
    plt.plot(freqs[:half], fft_vals[:half], color="teal")
    plt.xlabel("Normalized frequency")
    plt.ylabel("Magnitude")
    plt.title(f"{fig_name}")
    plt.tight_layout()

    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {file_path}")
    plt.show()

    return freqs, fft_vals


for r in resultados:
    date = r["date"]
    plot_lightcurve_comparison(
        r["tempo_horas"], r["curva_luz_cme"], r["curva_luz_no_cme"], r["tempo_transito"],
        f"lightcurve_comparison_{WAVELENGTH_VALUE}A_{date}",
    )
    plot_residual(r["tempo_horas"], r["residuo"], f"residual_{WAVELENGTH_VALUE}A_{date}")
    plot_fft_spectrum(r["residuo"], f"fft_residual_{WAVELENGTH_VALUE}A_{date}")


## Análise combinada entre eventos (correlação, PCA e FFT)

Réplica exata do bloco final de `main-fft-pca-sun-signals.ipynb` (`ruidos = [ruido, ruido_2, ruido_3, ruido_4]`), agora com os 4 resíduos do comprimento de onda selecionado.

In [ ]:
def analyze_residual_ensemble(resultados, fig_name, output_dir=OUTPUT_DIR):
    """
    Replicate the 171 A ensemble analysis: pairwise cross-correlation,
    PCA projection and FFT spectra across all event residuals.
    """
    residuos = [r["residuo"] for r in resultados]
    labels = [r["date"] for r in resultados]

    # 1) Pairwise cross-correlation
    print("Pairwise cross-correlation (normalized peak):")
    for i in range(len(residuos)):
        for j in range(i + 1, len(residuos)):
            corr = correlate(residuos[i], residuos[j], mode="full")
            max_corr = np.max(np.abs(corr)) / len(residuos[i])
            print(f"  {labels[i]} vs {labels[j]}: {max_corr:.3f}")

    # 2) PCA across residuals
    X = np.vstack(residuos)
    X_std = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_std)
    print("Explained variance ratio:", pca.explained_variance_ratio_)

    # 3) FFT spectra
    freqs = np.fft.fftfreq(len(residuos[0]))
    fft_results = [np.abs(np.fft.fft(r)) for r in residuos]
    half = len(freqs) // 2

    # 4) Combined figure
    fig = plt.figure(figsize=(15, 10))

    ax1 = fig.add_subplot(2, 2, 1)
    for label, r in zip(labels, residuos):
        ax1.plot(r, label=label)
    ax1.set_title(f"Residual signals (time domain) - AIA {WAVELENGTH_VALUE} A")
    ax1.legend()

    ax2 = fig.add_subplot(2, 2, 2)
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    for i, label in enumerate(labels):
        ax2.scatter(X_pca[i, 0], X_pca[i, 1], s=100, color=colors[i % len(colors)], label=label)
    ax2.set_title(f"PCA projection (2D) of residuals - AIA {WAVELENGTH_VALUE} A")
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")
    ax2.legend()

    ax3 = fig.add_subplot(2, 1, 2)
    for label, fft_vals in zip(labels, fft_results):
        ax3.plot(freqs[:half], fft_vals[:half], label=label)
    ax3.set_title(f"FFT spectrum of residual signals - AIA {WAVELENGTH_VALUE} A")
    ax3.set_xlabel("Normalized frequency")
    ax3.set_ylabel("Magnitude")
    ax3.legend()

    plt.tight_layout()

    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {file_path}")
    plt.show()

    return X_pca, freqs, fft_results


X_pca_result, freqs_result, fft_results = analyze_residual_ensemble(
    resultados, f"ensemble_analysis_{WAVELENGTH_VALUE}A_4-events"
)


## Salvando os arrays para comparação futura entre comprimentos de onda

Permite carregar os resíduos e o eixo de tempo em um notebook posterior e compará-los diretamente com os resíduos de outros canais (mesmos 4 eventos), sem precisar re-executar as simulações de trânsito.

In [ ]:
for r in resultados:
    date = r["date"]
    np.save(os.path.join(OUTPUT_DIR, f"residual_{WAVELENGTH_VALUE}A_{date}.npy"), r["residuo"])
    np.save(os.path.join(OUTPUT_DIR, f"tempo_horas_{WAVELENGTH_VALUE}A_{date}.npy"), r["tempo_horas"])

print(f"Residual and time arrays saved for a future cross-wavelength comparison (4 events, {WAVELENGTH_VALUE} A).")
